<a href="https://colab.research.google.com/github/oni-swr/bvh-mocap-via-video/blob/collab/eccv22_demo/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
from pathlib import Path
import math
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

path='../model'
#p = Path(path)
#if p.exists():
#    print(p.read_text())
#model = tf.saved_model.load('../model')  # Takes about 3 minutes






Found GPU at: /device:GPU:0


In [ ]:
model = hub.load('https://bit.ly/metrabs_l')  # Takes about 3 minutes


In [ ]:
img = tf.image.decode_image(tf.io.read_file('/content/images/2025-10-29 11-03-28.mkv'))

In [ ]:

pred = model.detect_poses(img, skeleton='smpl+head_30')
pred['poses3d'].shape

In [ ]:
def frame_gen(path):
    for frame in iio.imiter(path):  # HxWx3 uint8 RGB
        yield frame

def resize_to_384(img):
    img = tf.image.resize_with_pad(img, 384, 384, method="bilinear")
    return tf.cast(tf.round(img), tf.uint8)

ds = tf.data.Dataset.from_generator(
    lambda: frame_gen(video_path),
    output_signature=tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
).map(resize_to_384, num_parallel_calls=tf.data.AUTOTUNE)

batch_size = 32  # try 16/32/64; increase until you hit GPU OOM, then back off
frame_batches = ds.batch(batch_size, drop_remainder=False).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
H = W = 384
fov_deg = 55.0
fx = fy = 0.5 * W / math.tan(0.5 * math.radians(fov_deg))
cx, cy = W / 2.0, H / 2.0
K = tf.constant([[fx, 0.0, cx],
                 [0.0, fy, cy],
                 [0.0, 0.0, 1.0]], dtype=tf.float32)  # [3,3]

In [ ]:
def plot_results(image, pred, joint_names, joint_edges):
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    from matplotlib.patches import Rectangle
    fig = plt.figure(figsize=(10, 5.2))
    image_ax = fig.add_subplot(1, 2, 1)
    image_ax.imshow(image.numpy())
    for x, y, w, h, c in pred['boxes'].numpy():
        image_ax.add_patch(Rectangle((x, y), w, h, fill=False))

    pose_ax = fig.add_subplot(1, 2, 2, projection='3d')
    pose_ax.view_init(5, -75)
    pose_ax.set_xlim3d(-1500, 1500)
    pose_ax.set_zlim3d(-1500, 1500)
    pose_ax.set_ylim3d(2000, 5000)
    poses3d = pred['poses3d'].numpy()
    poses3d[..., 1], poses3d[..., 2] = poses3d[..., 2], -poses3d[..., 1]
    for pose3d, pose2d in zip(poses3d, pred['poses2d'].numpy()):
        for i_start, i_end in joint_edges:
            image_ax.plot(*zip(pose2d[i_start], pose2d[i_end]), marker='o', markersize=2)
            pose_ax.plot(*zip(pose3d[i_start], pose3d[i_end]), marker='o', markersize=2)
        image_ax.scatter(*pose2d.T, s=2)
        pose_ax.scatter(*pose3d.T, s=2)

In [ ]:
skeleton = "smpl+head_30"

@tf.function  # keep jit_compile=False by default; XLA can help on some GPUs but benchmark first
def run_batch(images):
    B = tf.shape(images)[0]
    K_batched = tf.repeat(K[tf.newaxis, :, :], repeats=B, axis=0)  # [B,3,3]
    return model.detect_poses_batched(
        images,
        intrinsic_matrix=K_batched,
        skeleton=skeleton,
        internal_batch_size=tf.constant(8, tf.int32),  # tune this
    )



In [ ]:
all_poses = []
for batch in frame_batches:
    with tf.device("/GPU:0"):  # ensures kernels run on the GPU when available
        pred = run_batch(batch)
    poses3d = pred["poses3d"]  # Ragged [B, (num_persons_i), J, 3]
    all_poses.extend(poses3d.to_list())

print(f"Processed frames: {len(all_poses)}")

In [ ]:
joint_names = model.per_skeleton_joint_names['smpl+head_30'].numpy().astype(str)
joint_edges = model.per_skeleton_joint_edges['smpl+head_30'].numpy()
plot_results(img, pred, joint_names, joint_edges)